# RetNet — a toy-scale build of Retentive Networks

A minimal implementation of **Retention**, from Sun et al., *"Retentive
Network: A Successor to Transformer for Large Language Models"* (2023).

Companion write-up: `README.md` in this folder.

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 1. The idea

RetNet is another **linear-attention-with-a-state** layer, in the same
family as GLA and KDA (see `../gla` and `../kda`) — a fixed-size state
matrix, updated one token at a time, read out with a query.

Where it differs: the decay applied to the state at every step is **not
predicted from the input at all**. It's a **fixed constant**, chosen before
training and never changed — one constant per attention head:

```
S_t = gamma_h * S_{t-1} + k_t (x) v_t     # gamma_h is fixed, not learned per-token
o_t = q_t^T S_t
```

No gate network computes `gamma`; it's just a number baked into the model
per head, e.g. `gamma_h = 1 - 2^(-5-h)`. Different heads get different
`gamma` values — some decay fast (short memory, good at local patterns),
some decay very slowly (long memory, good at long-range dependencies). This
is called **multi-scale retention**: instead of one learned, input-dependent
decay rate (like GLA), RetNet spreads a fixed range of decay rates across
its heads and lets each head specialize.

**Why fix the decay instead of learning it?** A fixed decay makes the
recurrence easier to reformulate in three equivalent forms — a parallel
(matrix) form for training, a recurrent form for fast inference, and a
chunked form for long sequences — because the decay factors are known in
advance rather than depending on the data. That's RetNet's main pitch:
same asymptotic efficiency as a transformer for training, but constant-memory,
constant-time inference like an RNN.

> **Simplification used here:** the sequential recurrence is used directly
> instead of RetNet's parallel/chunked forms — same math, much easier to
> read, much slower.

In [ ]:
class RetNet(nn.Module):
    def __init__(self, d_model=64, n_heads=2, d_head=32):
        super().__init__()
        self.h, self.dh = n_heads, d_head
        inner = n_heads * d_head
        self.q_proj = nn.Linear(d_model, inner, bias=False)
        self.k_proj = nn.Linear(d_model, inner, bias=False)
        self.v_proj = nn.Linear(d_model, inner, bias=False)
        self.gate_proj = nn.Linear(d_model, inner, bias=True)
        self.out_proj = nn.Linear(inner, d_model, bias=False)
        self.out_norm = nn.GroupNorm(n_heads, inner)
        # fixed, multi-scale decay: one constant per head, never updated by gradient descent
        gammas = torch.tensor([1 - 2 ** (-5 - hh) for hh in range(n_heads)])
        self.register_buffer('gamma', gammas)

    def forward(self, x):
        B, T, D = x.shape
        H, Dh = self.h, self.dh
        q = self.q_proj(x).view(B, T, H, Dh)
        k = self.k_proj(x).view(B, T, H, Dh)
        v = self.v_proj(x).view(B, T, H, Dh)

        S = x.new_zeros(B, H, Dh, Dh)
        outs = []
        for t in range(T):
            k_t, v_t, q_t = k[:, t], v[:, t], q[:, t]
            g = self.gamma.view(1, H, 1, 1)                                     # fixed decay, not input-dependent
            S = g * S + k_t.unsqueeze(-1) * v_t.unsqueeze(-2)
            o_t = torch.einsum('bhd,bhde->bhe', q_t, S)
            outs.append(o_t)

        o = torch.stack(outs, dim=1)   # B,T,H,Dh
        o = self.out_norm(o.reshape(B * T, H * Dh)).reshape(B, T, H * Dh)
        gate = F.silu(self.gate_proj(x))                                        # swish gate (paper's choice)
        return self.out_proj(gate * o)

## 2. Assembling a tiny language model

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(norm + self.eps) * self.weight

class SwiGLU(nn.Module):
    def __init__(self, d, hidden_mult=2):
        super().__init__()
        h = d * hidden_mult
        self.Wg = nn.Linear(d, h, bias=False)
        self.Wu = nn.Linear(d, h, bias=False)
        self.Wd = nn.Linear(h, d, bias=False)
    def forward(self, x):
        return self.Wd(F.silu(self.Wg(x)) * self.Wu(x))

class TinyLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([RetNet(d_model) for _ in range(n_layers)])
        self.mlps = nn.ModuleList([SwiGLU(d_model) for _ in range(n_layers)])
        self.norms1 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.norms2 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        x = self.embed(idx)
        for blk, mlp, n1, n2 in zip(self.blocks, self.mlps, self.norms1, self.norms2):
            x = x + blk(n1(x))
            x = x + mlp(n2(x))
        return self.lm_head(self.final_norm(x))

## Proving it actually works

Everything above is only worth something if gradients actually flow correctly
through RetNet once it's wired into a real model. So the rest of this
notebook:

1. wraps RetNet into a tiny 2-layer causal language model,
2. builds a **tiny synthetic dataset** (a repeating `"0123456789ABCDEF"`
   string — enough to check the model can learn *any* sequential structure
   at all, no real corpus needed),
3. runs **one forward + backward pass** as a sanity check (right output
   shape, no `NaN` gradients),
4. **trains for a few hundred steps**, and
5. **generates** from the trained model — if training worked, the output
   should show visible periodicity.

This is deliberately not a "real" training run. It exists purely to catch
architecture bugs, which is the whole point of a toy-scale build.

In [ ]:
# --- synthetic dataset ---
pattern = "0123456789ABCDEF"      # synthetic, no copyright concerns
text = pattern * 200
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
vocab_size = len(chars)
max_seq_len = 32

model = TinyLM(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built. Trainable parameters: {n_params:,}")

In [ ]:
# --- sanity check: one forward + backward pass before training ---
xb0 = data[:max_seq_len].unsqueeze(0).to(device)
yb0 = data[1:max_seq_len + 1].unsqueeze(0).to(device)
out0 = model(xb0)
logits0 = out0[0] if isinstance(out0, tuple) else out0
print(f"Sanity check -- logits shape: {tuple(logits0.shape)} (expect [1, {max_seq_len}, {vocab_size}])")
loss0 = F.cross_entropy(logits0.reshape(-1, vocab_size), yb0.reshape(-1))
if isinstance(out0, tuple):
    loss0 = loss0 + out0[1]
loss0.backward()
n_nan_grads = sum(torch.isnan(p.grad).any().item() for p in model.parameters() if p.grad is not None)
print(f"Sanity check -- initial loss: {loss0.item():.4f}, NaN grads: {n_nan_grads}")
model.zero_grad()

In [ ]:
# --- training loop ---
def get_batch(data, block_size, batch_size, device):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
n_steps, batch_size = 300, 16
print("Training on synthetic periodic sequence (verifies grads flow end-to-end)...")
for step in range(n_steps):
    xb, yb = get_batch(data, max_seq_len, batch_size, device)
    out = model(xb)
    logits = out[0] if isinstance(out, tuple) else out
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
    if isinstance(out, tuple):
        loss = loss + out[1]
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 50 == 0 or step == n_steps - 1:
        print(f"  step {step:4d} | loss {loss.item():.4f}")

In [ ]:
# --- generation ---
@torch.no_grad()
def generate(model, start_idx, n_new):
    model.eval()
    idx = start_idx.clone()
    for _ in range(n_new):
        out = model(idx)
        logits = out[0] if isinstance(out, tuple) else out
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return idx

start = data[:8].unsqueeze(0).to(device)
gen = generate(model, start, 48)[0].tolist()
print("Generated (should show visible periodicity if training worked):")
print(''.join(itos[i] for i in gen))

## Where to go from here

- **Make the decay learned instead of fixed** and you're back at GLA
  (`../gla`) — a good exercise in feeling out exactly what "selectivity"
  buys you over a fixed schedule.
- **Try the chunked recurrent form** — split the sequence into chunks,
  compute each chunk with the parallel (matrix) form, and carry the state
  between chunks with the recurrent form. This is RetNet's actual training
  recipe and a nice bridge between the two extremes.
- **Vary the number of heads and their `gamma` spread** and see how it
  changes what the toy model can learn — more heads means finer-grained
  multi-scale memory.

Reference: Sun, Dong, Huang, Ma, Xia, Xue, Wang, Wei, *"Retentive Network:
A Successor to Transformer for Large Language Models,"* 2023.